**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión**

---

# **Clase 21 - Métricas estadísticas aplicadas a los procesos organizacionales**

## Complemento

Este notebook se complementa con la presentación: **Metricas_Estadisticas.pdf**

Te recomendamos leer el PDF para trabajar con este notebook y tener una mejor comprensión de los conceptos teóricos.

## ¿Qué vamos a hacer en esta clase?

Un gerente te pide "un número que resuma cómo viene el negocio". Le das el promedio.
**Y el promedio te miente.**

Hoy aprendemos a describir un proceso organizacional sin que los números nos engañen.

| Parte | Tema | Pregunta que responde |
|---|---|---|
| **A** | El promedio y sus trampas | ¿Cuál es el valor "típico"? |
| **B** | Dispersión | ¿Qué tan parejo es el proceso? |
| **C** | Cuartiles y boxplot | ¿Cómo se reparten los valores? |
| **D** | Outliers | ¿Hay casos raros que ensucian todo? |
| **E** | Datos faltantes | ¿Qué hago con los agujeros? |
| **F** | Correlación | ¿Estas dos variables se mueven juntas? |
| **G** | Caso: controlar un proceso | ¿Está bajo control o se descontroló? |

> **La idea de fondo:** en gestión, **la variabilidad importa más que el promedio**. Un proveedor que
> entrega siempre en 5 días es mejor que uno que promedia 4 pero a veces tarda 15.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams["figure.figsize"] = (9, 4)
URL = "https://raw.githubusercontent.com/Datso653/Laboratorio-de-metodos-Cuantitativos-Aplicados-a-la-gestion/main/DF/"

ventas = pd.read_csv(URL + "ventas.csv")
ventas["Fecha"] = pd.to_datetime(ventas["Fecha"])
ventas["Total"] = ventas["Cantidad"] * ventas["Precio_Unitario"]

print("Operaciones:", len(ventas))
ventas.head(3)

---
# 🎯 Parte A — El promedio y sus trampas

Hay tres formas de responder *"¿cuánto es lo normal?"*, y **casi nunca coinciden**:

| Medida | Qué es | Comando |
|---|---|---|
| **Media** (promedio) | La suma dividida por la cantidad | `.mean()` |
| **Mediana** | El valor del medio si ordenás todo | `.median()` |
| **Moda** | El valor que más se repite | `.mode()` |

In [ ]:
print(f"Media   : $ {ventas['Total'].mean():>12,.0f}")
print(f"Mediana : $ {ventas['Total'].median():>12,.0f}")
print(f"Diferencia: {ventas['Total'].mean() / ventas['Total'].median() - 1:.1%} más alta la media")

**El promedio es 38% más alto que la mediana.** ¿Por qué?

Porque la media **se deja arrastrar por los valores extremos** y la mediana no. Unas pocas ventas
enormes de notebooks empujan el promedio hacia arriba, aunque la mayoría de las operaciones sean chicas.

> 🍕 **La analogía clásica:** si Messi entra a una pizzería con 10 personas adentro, el *patrimonio promedio*
> de los presentes pasa a ser de millones de dólares. La *mediana* casi no se mueve. ¿Cuál de los dos
> números describe mejor a la gente que está comiendo pizza?

In [ ]:
fig, ax = plt.subplots()
ax.hist(ventas["Total"], bins=30, color="#243b5e", edgecolor="white")
ax.axvline(ventas["Total"].mean(),   color="#e07b39", linewidth=2.5, label="Media")
ax.axvline(ventas["Total"].median(), color="#4caf50", linewidth=2.5, linestyle="--", label="Mediana")
ax.set_title("Distribución del monto de las ventas", loc="left", fontweight="bold")
ax.set_xlabel("Monto de la venta ($)")
ax.set_ylabel("Cantidad de operaciones")
ax.legend()
plt.tight_layout()
plt.show()

La distribución tiene una **cola larga hacia la derecha**: muchas ventas chicas y unas pocas muy grandes.
Se llama **distribución asimétrica** (o *sesgada*), y es la forma más común en datos de negocio:
ingresos, ventas, tiempos de espera, tamaños de empresa.

In [ ]:
asimetria = ventas["Total"].skew()   # skew: mide la asimetría

print(f"Coeficiente de asimetría: {asimetria:.3f}")
print()
print("Cómo leerlo:")
print("  ≈ 0    → simétrica    → media y mediana coinciden")
print("  > 0    → cola derecha → la media queda POR ENCIMA de la mediana   ← nuestro caso")
print("  < 0    → cola izquierda → la media queda POR DEBAJO de la mediana")

> 🎯 **Regla práctica para el trabajo:** si la distribución es asimétrica, **informá la mediana**.
> Si te piden sí o sí el promedio, informá los dos y explicá la diferencia. Eso es lo que hace la
> diferencia entre pasar un número y hacer análisis.

---
# 📏 Parte B — Dispersión: el número que nadie mira

Dos sucursales facturan en promedio lo mismo. Una lo hace todos los meses igual; la otra un mes explota
y al siguiente no vende nada. **Para gestionar, no son lo mismo en absoluto.**

| Medida | Qué mide | Comando |
|---|---|---|
| **Rango** | Máximo − mínimo | `.max() - .min()` |
| **Varianza** | Dispersión promedio al cuadrado | `.var()` |
| **Desvío estándar** | La raíz de la varianza (misma unidad que el dato) | `.std()` |
| **Coeficiente de variación (CV)** | Desvío ÷ media → **sin unidad** | `.std() / .mean()` |

In [ ]:
total = ventas["Total"]

print(f"Rango           : $ {total.max() - total.min():>12,.0f}")
print(f"Varianza        :   {total.var():>14,.0f}   ← en pesos AL CUADRADO, no se interpreta")
print(f"Desvío estándar : $ {total.std():>12,.0f}   ← en pesos, sí se interpreta")
print(f"CV              :   {total.std() / total.mean():>14.1%}   ← comparable entre variables distintas")

**El desvío estándar** se lee así: las ventas se apartan del promedio, típicamente, en unos $336.000.

**El coeficiente de variación (CV)** es el más útil para gestión porque **no tiene unidad**: permite
comparar la variabilidad de cosas distintas (pesos contra días contra unidades).

| CV | Interpretación en un proceso |
|---|---|
| < 10% | Muy estable ✅ |
| 10% – 30% | Variabilidad normal |
| > 30% | Proceso poco predecible ⚠️ |

Nuestro CV de ventas da **85%**: altísimo. Tiene sentido, porque estamos mezclando mouses de $5.000
con notebooks de $150.000 en la misma bolsa.

In [ ]:
# La pregunta de gestión: ¿qué sucursal es más predecible?
por_ciudad = ventas.groupby("Ciudad")["Total"].agg(["count", "mean", "median", "std"])
por_ciudad["CV"] = (por_ciudad["std"] / por_ciudad["mean"])
por_ciudad = por_ciudad.sort_values("CV")

por_ciudad.style.format({
    "mean": "${:,.0f}", "median": "${:,.0f}", "std": "${:,.0f}", "CV": "{:.1%}"
})

Mirá lo que aparece: **La Plata y Rosario facturan promedios parecidos, pero Rosario es mucho más
volátil** (CV 97% contra 73%). Si tuvieras que poner stock de seguridad en una sola sucursal,
ya sabés en cuál.

**Ese es el tipo de conclusión que el promedio solo, nunca te habría dado.**

---
# 📦 Parte C — Cuartiles y boxplot

Los **cuartiles** parten los datos ordenados en cuatro grupos iguales:

- **Q1** (percentil 25): el 25% de las ventas está por debajo.
- **Q2** (percentil 50): la **mediana**.
- **Q3** (percentil 75): el 75% está por debajo.
- **IQR** = Q3 − Q1: el **rango intercuartílico**, donde vive el 50% central de los datos.

In [ ]:
q1 = total.quantile(0.25)
q2 = total.quantile(0.50)
q3 = total.quantile(0.75)
iqr = q3 - q1

print(f"Q1  (25%) : $ {q1:>12,.0f}")
print(f"Q2  (50%) : $ {q2:>12,.0f}   ← mediana")
print(f"Q3  (75%) : $ {q3:>12,.0f}")
print(f"IQR       : $ {iqr:>12,.0f}   ← el 50% del medio se mueve en este rango")

In [ ]:
# El boxplot dibuja exactamente esos cinco números
fig, ax = plt.subplots(figsize=(9, 4))
datos = [g["Total"].values for _, g in ventas.groupby("Ciudad")]
etiquetas = sorted(ventas["Ciudad"].unique())

bp = ax.boxplot(datos, labels=etiquetas, patch_artist=True, vert=True)
for caja in bp["boxes"]:
    caja.set_facecolor("#243b5e")
    caja.set_alpha(0.7)
for mediana in bp["medians"]:
    mediana.set_color("#e07b39")
    mediana.set_linewidth(2)

ax.set_title("Distribución del monto de venta por ciudad", loc="left", fontweight="bold")
ax.set_ylabel("Monto ($)")
plt.tight_layout()
plt.show()

### Cómo se lee un boxplot

```
        │                    ← bigote superior: hasta Q3 + 1,5·IQR
    ┌───┴───┐
    │       │                ← Q3 (75%)
    ├───────┤                ← MEDIANA (línea naranja)
    │       │                ← Q1 (25%)
    └───┬───┘
        │                    ← bigote inferior
        ○                    ← puntos sueltos = OUTLIERS
```

Es el gráfico más denso en información de toda la estadística descriptiva: en un centímetro cuadrado
te muestra el centro, la dispersión, la asimetría y los casos raros. **Si la mediana no está en el medio
de la caja, la distribución es asimétrica.**

---
# 🚨 Parte D — Outliers: los casos raros

Un **outlier** (o valor atípico) es una observación que se aparta mucho del resto. La regla más usada
para detectarlos es la del **1,5 × IQR**:

$$\text{límite inferior} = Q_1 - 1{,}5 \cdot IQR \qquad \text{límite superior} = Q_3 + 1{,}5 \cdot IQR$$

Todo lo que caiga afuera de esos límites es sospechoso.

In [ ]:
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

outliers = ventas[(ventas["Total"] < lim_inf) | (ventas["Total"] > lim_sup)]

print(f"Límite inferior : $ {lim_inf:>12,.0f}")
print(f"Límite superior : $ {lim_sup:>12,.0f}")
print(f"Outliers detectados: {len(outliers)} de {len(ventas)}")

**Cero outliers.** No porque no haya ventas grandes, sino porque la distribución es tan dispersa
que el propio IQR es enorme. La regla del 1,5·IQR es conservadora con datos muy asimétricos.

Probemos con un dataset donde sí aparecen: **HMEQ**, una base real de 5.960 solicitudes de crédito
hipotecario de un banco.

In [ ]:
creditos = pd.read_csv(URL + "hmeq.csv")

print("Solicitudes:", len(creditos))
print("\nVariables:")
print("  LOAN     = monto solicitado")
print("  VALUE    = valor de la propiedad")
print("  DEBTINC  = relación deuda / ingreso")
print("  BAD      = 1 si el cliente entró en mora")
creditos[["LOAN", "VALUE", "DEBTINC"]].describe().round(1)

In [ ]:
def detectar_outliers(serie):
    """Devuelve cuántos outliers tiene una serie según la regla del 1,5 · IQR."""
    s = serie.dropna()                       # dropna: ignorar los faltantes
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    RIC = Q3 - Q1
    fuera = (s < Q1 - 1.5 * RIC) | (s > Q3 + 1.5 * RIC)
    return pd.Series({
        "media": s.mean(),
        "mediana": s.median(),
        "outliers": fuera.sum(),
        "% del total": 100 * fuera.mean(),
        "máximo": s.max(),
    })

resumen = pd.DataFrame({col: detectar_outliers(creditos[col])
                        for col in ["LOAN", "VALUE", "DEBTINC"]}).T
resumen.round(1)

Mirá `VALUE`: la mediana de las propiedades es de **89.000** dólares, pero el máximo es de **855.909**.
Hay 320 propiedades atípicamente caras.

### ⚠️ Un outlier NO se borra automáticamente

Antes de tocarlo, hay que preguntarse **de dónde salió**:

| Origen | Qué hacer |
|---|---|
| **Error de carga** (un cero de más) | Corregir o eliminar ✅ |
| **Caso real pero excepcional** (una mansión) | **Conservar** — es información valiosa |
| **Otra población mezclada** (mayoristas entre minoristas) | Separar en dos análisis |

> 🎯 En detección de fraude o de fallas, **el outlier es justamente lo que estás buscando**.
> Borrarlo sería tirar el hallazgo a la basura.

---
# 🕳️ Parte E — Datos faltantes

En datos reales **siempre** faltan cosas. Ignorarlos no los hace desaparecer: los convierte en un
sesgo silencioso.

In [ ]:
faltantes = pd.DataFrame({
    "nulos": creditos.isnull().sum(),
    "% del total": (100 * creditos.isnull().mean()).round(1)
})
faltantes[faltantes["nulos"] > 0].sort_values("nulos", ascending=False)

`DEBTINC` (relación deuda/ingreso) tiene **1.267 faltantes: el 21%**. Y no es un dato cualquiera:
es probablemente **la variable más importante** para decidir un crédito.

**La pregunta clave no es "¿con qué lo relleno?" sino "¿por qué falta?".** Si el dato falta más seguido
en los clientes que después entraron en mora, entonces **el hecho de que falte ya es información**.

In [ ]:
# Comprobémoslo: ¿la tasa de mora es distinta según falte o no el dato?
creditos["falta_debtinc"] = creditos["DEBTINC"].isnull()

comparacion = creditos.groupby("falta_debtinc")["BAD"].agg(["count", "mean"])
comparacion.columns = ["solicitudes", "tasa_de_mora"]
comparacion.index = ["Tiene el dato", "Le falta el dato"]

comparacion.style.format({"tasa_de_mora": "{:.1%}"})

**Ahí está.** Los casos donde falta el dato tienen una tasa de mora muchísimo más alta.
Si hubiéramos rellenado esos huecos con el promedio, habríamos **borrado la señal más fuerte del dataset**.

> 📌 **La moraleja:** antes de imputar un faltante, chequeá si la *ausencia* del dato está relacionada
> con lo que querés predecir. Se llama "faltante no aleatorio" y es una trampa en la que caen
> analistas con años de experiencia.

---
# 🔗 Parte F — Correlación

La **correlación de Pearson** mide si dos variables se mueven juntas. Va de −1 a +1.

| Valor | Significa |
|---|---|
| **+1** | Cuando una sube, la otra sube, siempre |
| **0** | No hay relación lineal |
| **−1** | Cuando una sube, la otra baja, siempre |

In [ ]:
corr = ventas[["Cantidad", "Precio_Unitario", "Total"]].corr()

corr.style.background_gradient(cmap="RdYlGn", vmin=-1, vmax=1).format("{:.3f}")

Lo interesante está en la esquina: **Cantidad y Precio_Unitario tienen correlación 0,098** — o sea,
prácticamente cero. En este negocio, que un producto sea caro **no** hace que se venda en menor cantidad.

Y las dos tienen correlación ~0,70 con el Total, lo cual es obvio: el total es literalmente el producto
de las dos.

### ⚠️ Las tres trampas de la correlación

**1. Correlación no es causalidad.** El consumo de helado y los ahogamientos correlacionan fuerte.
No es que el helado ahogue: hay una tercera variable, el calor, que mueve a las dos.

**2. Solo detecta relaciones *lineales*.** Dos variables pueden tener una relación perfecta en forma
de U y dar correlación cero.

**3. Los outliers la distorsionan muchísimo.** Un solo punto extremo puede inventar una correlación
que no existe, o tapar una que sí.

In [ ]:
# Demostración de la trampa 2: relación perfecta, correlación cero
x = np.linspace(-10, 10, 100)
y = x ** 2                       # una parábola: relación perfecta pero NO lineal

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, color="#243b5e", s=18)
ax.set_title(f"Relación perfecta, correlación = {np.corrcoef(x, y)[0,1]:.3f}",
             loc="left", fontweight="bold")
ax.set_xlabel("x"); ax.set_ylabel("y = x²")
plt.tight_layout()
plt.show()

print("Moraleja: SIEMPRE graficá antes de confiar en un coeficiente.")

---
# 🏭 Parte G — Caso: ¿el proceso está bajo control?

Acá juntamos todo. Un supermercado registró **12 semanas** de operación. La pregunta del dueño es
simple: *"¿el negocio viene estable o se me está descontrolando?"*

In [ ]:
sup = pd.read_csv(URL + "datos_supermercado_simulados.csv")

# Ingreso semanal = suma de cantidad × precio de cada rubro
sup["ingreso"] = (sup["cant_alimentos"] * sup["precio_alimentos"] +
                  sup["cant_bebidas"]   * sup["precio_bebidas"] +
                  sup["cant_limpieza"]  * sup["precio_limpieza"])
sup["margen"] = sup["ingreso"] - sup["costos_operativos"]

sup[["semana", "ingreso", "costos_operativos", "margen"]]

In [ ]:
for concepto in ["ingreso", "costos_operativos", "margen"]:
    s = sup[concepto]
    print(f"{concepto:20s} media $ {s.mean():>10,.0f} | desvío $ {s.std():>9,.0f} | CV {s.std()/s.mean():>6.1%}")

El ingreso tiene un CV del **5%** y los costos del **6%**: los dos son procesos muy estables.
Pero el **margen** —que es la resta de ambos— tiene un CV mucho más alto.

> 💡 **Esto es clave en gestión:** la variabilidad de una diferencia es **mayor** que la de sus partes.
> Por eso los márgenes son siempre más volátiles que las ventas, y por eso una empresa con costos
> altos sufre mucho más cualquier sacudón.

### El gráfico de control

La herramienta clásica de control de procesos. Se dibujan tres líneas:

- La **media** del proceso.
- Dos **límites de control** en media ± 3 desvíos.

Si todos los puntos caen adentro, el proceso está **bajo control**: lo que se ve es variación normal.
Un punto afuera es una señal de que **algo cambió** y hay que ir a investigar.

In [ ]:
serie = sup["ingreso"]
media, desvio = serie.mean(), serie.std()
lim_sup_c, lim_inf_c = media + 3 * desvio, media - 3 * desvio

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(sup["semana"], serie, marker="o", color="#243b5e", linewidth=2, label="Ingreso semanal")
ax.axhline(media,     color="#4caf50", linestyle="-",  label=f"Media (${media:,.0f})")
ax.axhline(lim_sup_c, color="#e07b39", linestyle="--", label="Límites de control (±3σ)")
ax.axhline(lim_inf_c, color="#e07b39", linestyle="--")
ax.fill_between(sup["semana"], lim_inf_c, lim_sup_c, color="#4caf50", alpha=0.06)

ax.set_title("Gráfico de control — ingreso semanal del supermercado", loc="left", fontweight="bold")
ax.set_xlabel("Semana"); ax.set_ylabel("Ingreso ($)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

fuera = ((serie > lim_sup_c) | (serie < lim_inf_c)).sum()
print(f"Semanas fuera de control: {fuera}  →  " +
      ("proceso ESTABLE ✅" if fuera == 0 else "hay que investigar ⚠️"))

---
# 📝 Ejercicios

**Ejercicio 1.** Calculá media, mediana y CV de la columna `Cantidad` de `ventas`.
¿Es una variable más estable o menos estable que `Total`? ¿Por qué te parece?

In [ ]:
# Tu respuesta acá

**Ejercicio 2.** Agrupá `ventas` por `Producto` y armá una tabla con media, mediana, desvío y CV.
¿Qué producto tiene el proceso de venta más predecible?

In [ ]:
# Tu respuesta acá

**Ejercicio 3.** Hacé un boxplot del `Total` por `Producto`. ¿Qué producto muestra más asimetría?

In [ ]:
# Tu respuesta acá

**Ejercicio 4.** En `creditos`, aplicá la regla del 1,5·IQR a la columna `CLAGE`
(antigüedad de la línea de crédito más vieja, en meses). ¿Cuántos outliers hay?
Mirá el máximo: ¿te parece un error de carga o un caso real?

In [ ]:
# Tu respuesta acá

**Ejercicio 5.** Calculá la matriz de correlación de `LOAN`, `VALUE`, `MORTDUE` y `DEBTINC` en `creditos`.
¿Cuál es el par más correlacionado? ¿Tiene sentido económico?

In [ ]:
# Tu respuesta acá

**Ejercicio 6 (integrador 🥇⚡🤓).** Armá un gráfico de control para el **margen** del supermercado.
Después respondé: si tuvieras que avisarle al dueño de un solo indicador para monitorear todas las semanas,
¿elegirías el ingreso, el costo o el margen? Justificá con los CV que calculamos.

In [ ]:
# Tu respuesta acá

---
## 🧭 Para llevarse

| Concepto | Comando | Cuándo lo usás |
|---|---|---|
| Media | `.mean()` | Datos simétricos |
| Mediana | `.median()` | **Datos asimétricos** (casi siempre en negocios) |
| Desvío estándar | `.std()` | Dispersión en la unidad original |
| **Coef. de variación** | `.std() / .mean()` | **Comparar variabilidad entre procesos** |
| Cuartiles | `.quantile([.25, .5, .75])` | Repartir la distribución |
| Asimetría | `.skew()` | Decidir si informar media o mediana |
| Outliers | Regla 1,5 · IQR | Detectar casos raros |
| Faltantes | `.isnull().sum()` | Antes de cualquier cálculo |
| Correlación | `.corr()` | Relación **lineal** entre variables |
| Resumen completo | `.describe()` | Primer vistazo |

**Las tres frases para llevarse:**
1. Si la distribución es asimétrica, **el promedio miente**.
2. En gestión, **la variabilidad importa más que el nivel**.
3. **Que un dato falte también es un dato.**